In [ ]:
import logging

from utils import utils

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

In [ ]:
spatial_extent = {
    "west": 30.5503711040000994,
    "south": 1.0709279050000799,
    "east": 31.2229521229999989,
    "north": 1.5469373050000299,
}

resample_spatial_resolution = 30  # m

temporal_variability_threshold = 0.5  # units: dB
flattening_threshold = 0.12  # units: dimensionless
logistic_sse_threshold = 18.3  # units: dB^2

min_connected_area = 10000  # m^2

In [ ]:
# more restrictive (excludes more pixels from being detected as deforestation) than default
temporal_variability_threshold = 0.6
flattening_threshold = 0.14

# Script

In [ ]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

## Sentinel 1

In [ ]:
# load results from previous batch job
JOB_ID = "j-2607241322294e99a2aaac75a3fc3708"

s1_dB = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
process_graph_results.append(
    s1_dB.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0000_s1_dB",
        },
    )
)

In [ ]:
# load_stac adds a time dimension 😠
s1_dB = s1_dB.drop_dimension("t")

In [ ]:
process_graph_results.append(
    s1_dB.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0001_s1_dB",
        },
    )
)

# load forest baseline

In [ ]:
# load results from previous batch job
JOB_ID = "j-26072410014341f0b1905a575d801af1"

forest_baseline_mask = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0010_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_forest_baseline_mask",
        },
    )
)

In [ ]:
# load_stac adds a time dimension 😠
forest_baseline_mask = forest_baseline_mask.drop_dimension("t")

In [ ]:
# load_stac incorrectly sets nodata=0 ⚠️ 😠
forest_baseline_mask = forest_baseline_mask == 1

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0012_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0012_forest_baseline_mask",
        },
    )
)

In [ ]:
# resample onto S1 grid @ resample_spatial_resolution
forest_baseline_mask = forest_baseline_mask.resample_cube_spatial(
    s1_dB,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

In [ ]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0015_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0015_forest_baseline_mask",
        },
    )
)

## temporal variability mask

In [ ]:
# mask of interesting pixels

# band math
temporal_variability_mask = s1_dB.band("sd") >= temporal_variability_threshold

In [ ]:
process_graph_results.append(
    temporal_variability_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0020_temporal_variability_mask",
        },
    )
)
process_graph_results.append(
    temporal_variability_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0020_temporal_variability_mask",
        },
    )
)

# flattening mask

In [ ]:
# TODO: ATBD says denominator is abs(gamma_max)
# but `2_deforesation_tracker.ipynb` uses p05
# Dascalu 2023 has abs(gamma_max)


def flattening_reducer(
    bands: openeo.processes.ProcessBuilder,
) -> openeo.processes.ProcessBuilder:
    p05 = bands.array_element(label="p05")
    p95 = bands.array_element(label="p95")
    return (p95 - p05) / p05.absolute()


flattening = s1_dB.reduce_bands(flattening_reducer)

In [ ]:
assert flattening._in_bandmath_mode()

In [ ]:
process_graph_results.append(
    flattening.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0030_flattening",
        },
    )
)
process_graph_results.append(
    flattening.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0030_flattening",
        },
    )
)

In [ ]:
# mask of good pixels where flattening >= flattening_threshold

# band math
flattening_mask = flattening >= flattening_threshold

In [ ]:
process_graph_results.append(
    flattening_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0040_flattening_mask",
        },
    )
)
process_graph_results.append(
    flattening_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0040_flattening_mask",
        },
    )
)

# SSE goodness of fit mask

In [ ]:
sse_threshold_mask = s1_dB.band("min_sse") <= logistic_sse_threshold

In [ ]:
process_graph_results.append(
    sse_threshold_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0050_sse_threshold_mask",
        },
    )
)
process_graph_results.append(
    sse_threshold_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_sse_threshold_mask",
        },
    )
)

# Combine all masks

In [ ]:
# forest_baseline_mask.band("B0") & temporal_variability_mask & flattening_mask & sse_threshold_mask
# BandMathException: 'Band math' between bands of different data cubes is not supported yet. 🙁

In [ ]:
# in order to do band math, we need to stack all of the inputs into a single DataCube
# https://github.com/Open-EO/openeo-python-client/issues/748

b1 = forest_baseline_mask.rename_labels(
    dimension="bands", target=["forest_baseline_mask"]
)
b2 = temporal_variability_mask.add_dimension(
    "bands", label="temporal_variability_mask", type="bands"
)
b3 = flattening_mask.add_dimension("bands", label="flattening_mask", type="bands")
b4 = sse_threshold_mask.add_dimension("bands", label="sse_threshold_mask", type="bands")

combined_masks = b1.merge_cubes(b2).merge_cubes(b3).merge_cubes(b4)

In [ ]:
process_graph_results.append(
    combined_masks.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0060_combined_masks",
        },
    )
)
process_graph_results.append(
    combined_masks.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_combined_masks",
        },
    )
)
# check dtype of the merge_cubes bands - uint8

In [ ]:
# deforestation event = 1
deforestation_event_mask = (
    combined_masks.band("temporal_variability_mask")
    & combined_masks.band("flattening_mask")
    & combined_masks.band("forest_baseline_mask")
    & combined_masks.band("sse_threshold_mask")
)

In [ ]:
process_graph_results.append(
    deforestation_event_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0061_deforestation_event_mask",
        },
    )
)
process_graph_results.append(
    deforestation_event_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0061_deforestation_event_mask",
        },
    )
)

In [ ]:
# ⚠️ there are some horrible artifacts in this mask
# https://forum.dataspace.copernicus.eu/t/load-stac-configure-nodata/5267
# in tiles over water

inverse_deforestation_event_mask = utils.logical_not(deforestation_event_mask)

In [ ]:
process_graph_results.append(
    inverse_deforestation_event_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0062_inverse_deforestation_event_mask",
        },
    )
)
process_graph_results.append(
    inverse_deforestation_event_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0062_inverse_deforestation_event_mask",
        },
    )
)

# Apply deforestation event mask to `min_sse_t`

In [ ]:
min_sse_t_masked = s1_dB.band("min_sse_t").mask(inverse_deforestation_event_mask)

In [ ]:
process_graph_results.append(
    min_sse_t_masked.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0070_min_sse_t_masked",
        },
    )
)
process_graph_results.append(
    min_sse_t_masked.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_min_sse_t_masked",
        },
    )
)

## mask based on connectivity

of the natural forest remaining, are regions of forest too small to meet the minimum connected area threshold?

In [ ]:
remaining_forest_mask = (
    combined_masks.band("forest_baseline_mask") & inverse_deforestation_event_mask
)

In [ ]:
process_graph_results.append(
    remaining_forest_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0080_remaining_forest_mask",
        },
    )
)
process_graph_results.append(
    remaining_forest_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0080_remaining_forest_mask",
        },
    )
)

In [ ]:
connectivity_udf = openeo.UDF.from_file(
    "../udf/connectivity_mask.py",
    runtime="Python",
    version="3.11",
    context={
        "spatial_resolution": resample_spatial_resolution,
        "min_connected_area": min_connected_area,
    },
)

In [ ]:
# mask where 1 = small region to be excluded
small_region_mask = remaining_forest_mask.apply_neighborhood(
    connectivity_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0090_small_region_mask",
        },
    )
)

In [ ]:
# apply_neighborhood UDF seems to return float32, even if it's a mask
# data types: https://github.com/locationtech/geotrellis/blob/master/raster/src/main/scala/geotrellis/raster/CellType.scala
small_region_mask = small_region_mask.convert_data_type("bool")

In [ ]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0091_small_region_mask",
        },
    )
)

In [ ]:
# stack min_sse_t_masked + small_region_mask into a single datacube
# in preparation for the nearest neighbour fill UDF

_data = min_sse_t_masked.add_dimension("bands", label="data", type="bands")
_mask = small_region_mask.add_dimension("bands", label="mask", type="bands")
combined = _data.merge_cubes(_mask)

In [ ]:
process_graph_results.append(
    combined.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0100_combined",
        },
    )
)
process_graph_results.append(
    combined.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0100_combined",
        },
    )
)

In [ ]:
nearest_neighbour_fill_udf = openeo.UDF.from_file(
    "../udf/nearest_neighbour_fill.py",
    runtime="Python",
    version="3.11",
)

In [ ]:
min_sse_t = combined.apply_neighborhood(
    nearest_neighbour_fill_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
process_graph_results.append(
    min_sse_t.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0110_min_sse_t",
        },
    )
)
process_graph_results.append(
    min_sse_t.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0110_min_sse_t",
        },
    )
)

In [ ]:
# give it a time dimension, otherwise openEO writes malformed STAC
min_sse_t = min_sse_t.add_dimension("t", label="2026-01-01T00:00:00Z", type="temporal")

In [ ]:
process_graph_results.append(
    min_sse_t.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0111_min_sse_t",
        },
    )
)

# Run batch job

In [ ]:
multi_result = openeo.MultiResult(process_graph_results)

In [ ]:
job = multi_result.create_job()
job.start_and_wait()
# Inspect job.logs() if it fails

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-script/
!rm -r output-script/

In [ ]:
results.download_files("output-script/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)